# 파주시 H3 해상도 7·8·9 테스트

파주시(운정·파주) 대상 광역버스의 정류장 좌표를 H3 해상도 7·8·9에 각각 배정하고, 셀별 정류장·노선 분포를 화면에서 비교합니다.

이 노트북은 테스트용이며 결과 파일을 저장하지 않습니다.

### ?? ? 1. ??????????? ?? ?? ??


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

try:
    import h3
except ImportError as e:
    raise ImportError("h3 패키지가 필요합니다. VS Code 터미널에서 pip install h3 실행 후 다시 실행하세요.") from e

BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / "gtx_a_seoul_bus_outputs"
CACHE_DIR = BASE_DIR / "api_cache_gtx_a_routes"
CANDIDATE_FILE = OUTPUT_DIR / "gtx_a_seoul_metropolitan_bus_candidates.csv"

print("작업 폴더:", BASE_DIR)
print("캐시 폴더 존재 여부:", CACHE_DIR.exists())

### ?? ? 2. ?? ?? ?? ID ??


In [ ]:
# 파주시(운정·파주) 대상 노선 ID 추출
candidates = pd.read_csv(CANDIDATE_FILE, dtype=str, encoding="utf-8-sig")
paju_routes = candidates[
    candidates["GTX_A_역"].eq("운정중앙역")
    | candidates["대상지역"].fillna("").str.contains("파주")
].copy()
paju_route_ids = set(paju_routes["노선ID"].dropna().astype(str).str.strip())
paju_route_numbers = sorted(paju_routes["버스번호"].dropna().astype(str).str.strip().str.lstrip("'").unique())

print("파주시 대상 노선 ID 수:", len(paju_route_ids))
print("파주시 대상 버스번호:", paju_route_numbers)
display(paju_routes[["GTX_A_역", "대상지역", "버스번호", "노선ID"]].drop_duplicates())

### ?? ? 3. API ???? ??? ?? ??


In [ ]:
# 기존 API 캐시에서 파주시 대상 노선의 정류장 좌표 읽기
stop_records = []
cache_files = list(CACHE_DIR.glob("*.json"))

for cache_file in cache_files:
    try:
        payload = json.loads(cache_file.read_text(encoding="utf-8"))
        response = payload.get("response", payload.get("Response", payload))
        body = response.get("body", {}) or {}
        items = body.get("items", {}) or {}
        items = items.get("item", []) if isinstance(items, dict) else items
        if isinstance(items, dict):
            items = [items]
        for item in items or []:
            route_id = str(item.get("routeid", "")).strip()
            lat = item.get("gpslati")
            lng = item.get("gpslong")
            if route_id in paju_route_ids and lat is not None and lng is not None:
                stop_records.append({
                    "route_id": route_id,
                    "stop_id": str(item.get("nodeid", "")).strip(),
                    "stop_no": item.get("nodeno"),
                    "latitude": float(lat),
                    "longitude": float(lng),
                    "stop_order": item.get("nodeord"),
                })
    except Exception:
        continue

stops = pd.DataFrame(stop_records).drop_duplicates(subset=["route_id", "stop_id"])
print("읽은 캐시 파일 수:", len(cache_files))
print("파주시 대상 노선 정류장 기록 수:", len(stops))
print("고유 정류장 수:", stops["stop_id"].nunique() if not stops.empty else 0)
display(stops.head())

### ?? ? 4. ??? H3 ?? ???? ?? ??


In [ ]:
def to_h3_cell(latitude, longitude, resolution):
    # h3-py 최신 버전과 구버전 모두 지원
    if hasattr(h3, "latlng_to_cell"):
        return h3.latlng_to_cell(latitude, longitude, resolution)
    return h3.geo_to_h3(latitude, longitude, resolution)

if stops.empty:
    print("정류장 좌표를 읽지 못했습니다. 노선ID 체계와 캐시 파일을 확인하세요.")
else:
    comparison = []
    for resolution in [7, 8, 9]:
        temp = stops.copy()
        temp["h3_cell"] = temp.apply(
            lambda row: to_h3_cell(row["latitude"], row["longitude"], resolution),
            axis=1,
        )
        cell_summary = (
            temp.groupby("h3_cell", as_index=False)
            .agg(
                정류장수=("stop_id", "nunique"),
                노선수=("route_id", "nunique"),
                평균위도=("latitude", "mean"),
                평균경도=("longitude", "mean"),
            )
        )
        comparison.append({
            "H3해상도": resolution,
            "정류장포함셀수": len(cell_summary),
            "셀당평균정류장수": cell_summary["정류장수"].mean(),
            "셀당최대정류장수": cell_summary["정류장수"].max(),
            "셀당평균노선수": cell_summary["노선수"].mean(),
            "셀당최대노선수": cell_summary["노선수"].max(),
        })
        print(f"\n[H3 해상도 {resolution}]")
        display(cell_summary.sort_values(["정류장수", "노선수"], ascending=False).head(20))

    comparison_df = pd.DataFrame(comparison)
    print("해상도 비교")
    display(comparison_df)
    print("참고: 이 테스트는 화면 표시만 하며 CSV·지도 파일을 저장하지 않습니다.")

### ?? ? 5. H3 ???? ??? ?? ???


In [ ]:
# H3 셀과 정류장 위치를 지도에 표시합니다. 지도 파일은 저장하지 않습니다.
try:
    import folium
except ImportError as e:
    raise ImportError("folium 패키지가 필요합니다. VS Code 터미널에서 pip install folium 실행 후 다시 실행하세요.") from e

if stops.empty:
    print("표시할 정류장 좌표가 없습니다. 앞 셀부터 순서대로 실행하세요.")
else:
    center = [stops["latitude"].mean(), stops["longitude"].mean()]

    for resolution, color in [(7, "red"), (8, "blue"), (9, "green")]:
        map_df = stops.copy()
        map_df["h3_cell"] = map_df.apply(
            lambda row: to_h3_cell(row["latitude"], row["longitude"], resolution),
            axis=1,
        )
        m = folium.Map(location=center, zoom_start=11, tiles="CartoDB positron")

        for cell in map_df["h3_cell"].unique():
            if hasattr(h3, "cell_to_boundary"):
                boundary = h3.cell_to_boundary(cell)
            else:
                boundary = h3.h3_to_geo_boundary(cell)
            folium.Polygon(
                locations=boundary,
                color=color,
                weight=2,
                fill=True,
                fill_color=color,
                fill_opacity=0.12,
                tooltip=f"H3 {resolution}: {cell}",
            ).add_to(m)

        for _, row in map_df.drop_duplicates("stop_id").iterrows():
            folium.CircleMarker(
                location=[row["latitude"], row["longitude"]],
                radius=3,
                color="black",
                fill=True,
                fill_color="yellow",
                fill_opacity=0.9,
                tooltip=f"정류장 ID: {row['stop_id']}",
            ).add_to(m)

        print(f"H3 해상도 {resolution} 지도")
        display(m)